In [ ]:
import numpy as np
import pandas as pd
import os

## Download data from Retrosheet

In [7]:
import requests

os.makedirs('../data/zips', exist_ok=True)

url = 'https://www.retrosheet.org/downloads/plays/2025plays.zip'
file_path = '../data/zips/2025plays.zip' # NOTE: Path for non-notebooks

try:
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, 'wb') as file:
        file.write(response.content)

except Exception as e:
    print(e)

### Extract the zip

In [10]:
import zipfile

with zipfile.ZipFile('../data/zips/2025plays.zip', 'r') as zip_ref:
    zip_ref.extractall('../data/plays')

## Load the DF

In [24]:
df = pd.read_csv('../data/plays/2025plays.csv')

/var/folders/fz/tl6wl4zn14d896djssv9plyr0000gn/T/ipykernel_22936/4099493405.py:1: DtypeWarning: Columns (0: umplf, 1: umprf) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/plays/2025plays.csv')


## Create RE Matrix

In [ ]:
def get_state(row):
    bases = ''
    
    bases += '-' if pd.isna(row['br1_pre']) else '1'
    bases += '-' if pd.isna(row['br2_pre']) else '2'
    bases += '-' if pd.isna(row['br3_pre']) else '3'
    
    if bases == '---':
        return 0 + (8 * row['outs_pre'])
    elif bases == '1--':
        return 1 + (8 * row['outs_pre'])
    elif bases == '-2-':
        return 2 + (8 * row['outs_pre'])
    elif bases == '--3':
        return 3 + (8 * row['outs_pre'])
    elif bases == '12-':
        return 4 + (8 * row['outs_pre'])
    elif bases == '1-3':
        return 5 + (8 * row['outs_pre'])
    elif bases == '-23':
        return 6 + (8 * row['outs_pre'])
    elif bases == '123':
        return 7 + (8 * row['outs_pre'])


    return 'XXX'


df_re = df[[
    'gid',
    'inning',
    'batter',
    'pitcher',
    'single',
    'double',
    'triple',
    'hr',
    'sh',
    'sf',
    'hbp',
    'walk',
    'k',
    'xi',
    'roe',
    'fc',
    'outs_pre',
    'br1_pre',
    'br2_pre',
    'br3_pre',
    'gametype'
]]

df_re['re_state'] = df_re.apply(get_state, axis=1)

df_re['re_state'].value_counts()
# For some reason there's a PA with 3 outs?
# df_re = df_re[df_re['re_state'] != 24]

,gid,inning,batter,pitcher,single,double,triple,hr,sh,sf,...,k,xi,roe,fc,outs_pre,br1_pre,br2_pre,br3_pre,gametype,re_state
192742,LAN202510160,7,perkb002,treib001,0,0,0,0,0,0,...,0,0,0,0,3,NaN,NaN,NaN,lcs,24
